# Absenteeism — Phase 2: Logistic Regression

**Purpose:** build a model that predicts whether an employee will be *excessively*
absent (more than the median number of hours).

**Input:** `df_preprocessed.csv`
**Outputs:** pickled `model` and `scaler` files (used by Phase 3)

Fresh notebook by design — modeling is a separate concern from preprocessing.

## 0. Imports and load

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_columns = None

data_preprocessed = pd.read_csv('df_preprocessed.csv')
data_preprocessed.head()

## 1. Create the target

The model needs a category, not a number of hours. We split on the **median** so the
two classes are roughly balanced — a balanced target is what makes accuracy a
meaningful metric here. (Phase 1 confirmed the hours distribution is non-normal, which
is exactly why the median, not the mean, is the right cutoff.)

In [ ]:
data_preprocessed['Absenteeism Time in Hours'].median()

In [ ]:
targets = np.where(
    data_preprocessed['Absenteeism Time in Hours'] >
    data_preprocessed['Absenteeism Time in Hours'].median(), 1, 0)
data_preprocessed['Excessive Absenteeism'] = targets

In [ ]:
# check balance: how close to 50/50 are we?
targets.sum() / targets.shape[0]

## 2. Select inputs (with a checkpoint)

We drop the target column plus three features that the course's backward elimination
showed carry almost no predictive weight: `Day of the Week`, `Daily Work Load Average`,
`Distance to Work`.

In [ ]:
data_with_targets = data_preprocessed.drop(
    ['Absenteeism Time in Hours', 'Day of the Week',
     'Daily Work Load Average', 'Distance to Work'], axis=1)
# confirm checkpoint (should be False = different object)
data_with_targets is data_preprocessed

In [ ]:
unscaled_inputs = data_with_targets.iloc[:, :-1]
unscaled_inputs.columns.values

## 3. Custom scaler — scale only the numerical columns

Standardizing dummy variables or binary flags is meaningless and distorts
interpretation. This CustomScaler wraps sklearn's StandardScaler but only touches the
columns we tell it to, leaving `Reason_*` and `Education` untouched.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns, copy=True, with_mean=True, with_std=True):
        self.scaler = StandardScaler(copy=copy, with_mean=with_mean, with_std=with_std)
        self.columns = columns
        self.mean_ = None
        self.var_ = None

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns], y)
        self.mean_ = np.mean(X[self.columns], axis=0)
        self.var_ = np.var(X[self.columns], axis=0)
        return self

    def transform(self, X, y=None, copy=None):
        init_col_order = X.columns
        X_scaled = pd.DataFrame(self.scaler.transform(X[self.columns]),
                                columns=self.columns, index=X.index)
        X_not_scaled = X.loc[:, ~X.columns.isin(self.columns)]
        return pd.concat([X_not_scaled, X_scaled], axis=1)[init_col_order]

In [ ]:
columns_to_omit = ['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Education']
columns_to_scale = [x for x in unscaled_inputs.columns if x not in columns_to_omit]

absenteeism_scaler = CustomScaler(columns_to_scale)
absenteeism_scaler.fit(unscaled_inputs)
scaled_inputs = absenteeism_scaler.transform(unscaled_inputs)
scaled_inputs.head()

## 4. Train/test split\n80/20, fixed random_state for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    scaled_inputs, targets, test_size=0.2, random_state=20)
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

## 5. Fit the model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

reg = LogisticRegression(max_iter=1000)
reg.fit(x_train, y_train)
reg.score(x_train, y_train)

## 6. Coefficients and odds ratios

The odds ratio = exp(coefficient). A value > 1 means the feature increases the odds of
excessive absenteeism; < 1 means it decreases them. This table is the real business
deliverable — it tells HR *which factors matter and by how much*.

In [ ]:
summary_table = pd.DataFrame(columns=['Feature name'],
                             data=unscaled_inputs.columns.values)
summary_table['Coefficient'] = np.transpose(reg.coef_)
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table['Odds_ratio'] = np.exp(summary_table.Coefficient)
summary_table.sort_values('Odds_ratio', ascending=False)

**Reading the result:** the reason groups dominate. `Reason_3` (injuries/external
causes) and `Reason_1` (diseases) raise the odds the most — intuitive, since these are
serious, unplanned absences. Transportation expense and number of children push odds up
mildly. Age, education and pets slightly lower them.

## 7. Evaluate on the test set

In [ ]:
reg.score(x_test, y_test)

In [ ]:
# enrichment beyond the course: confusion matrix + AUC
y_pred = reg.predict(x_test)
print('Confusion matrix:')
print(metrics.confusion_matrix(y_test, y_pred))
print()
print(metrics.classification_report(y_test, y_pred))
auc = metrics.roc_auc_score(y_test, reg.predict_proba(x_test)[:, 1])
print(f'ROC-AUC: {auc:.4f}')

In [ ]:
# probabilities for class 1 — these feed the Tableau analysis later
predicted_proba = reg.predict_proba(x_test)
predicted_proba[:, 1][:10]

## 8. Save model and scaler (pickle)

In [ ]:
import pickle

with open('model', 'wb') as file:
    pickle.dump(reg, file)

with open('scaler', 'wb') as file:
    pickle.dump(absenteeism_scaler, file)

print('model and scaler saved')

## Summary

- Target: median cutoff, ~46% positives (well balanced).
- Train accuracy ~0.77, test accuracy ~0.75, ROC-AUC ~0.78.
- Reason groups are by far the strongest predictors.
- `model` and `scaler` are pickled for deployment in Phase 3.

**Next:** Phase 3 — wrap this into a reusable module that scores new data.